In [4]:
import torch
import torch.nn as nn
import numpy as np
from scipy import stats
from collections import Counter
import sklearn
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from torch.utils.data import TensorDataset, DataLoader


In [5]:
## First we will simulate some data to do logisitc regression. 

def sigmoid(x):
    return(1/(1+np.exp(-x)))

num_feature = 5
num_samples = 1200
feature_name = [ f'feature_{i}' for i in range(num_feature)]
features = np.random.standard_normal((num_samples,num_feature))
true_weight_matrix = np.random.uniform(low=-1,high=1,size=num_feature)
true_beta = np.random.randn()



true_log_odds = features @ true_weight_matrix.reshape(5,1) + true_beta
true_prob = sigmoid(true_log_odds)
target_y = np.random.binomial(n=1, p = true_prob).squeeze()
#bernoulli_p = sigmoid()
Counter(target_y.tolist())

Counter({0: 984, 1: 216})

In [24]:
features.dtype

dtype('float64')

In [25]:
X_torch, Y_torch = torch.from_numpy(features), torch.from_numpy(target_y)
X_torch = X_torch.float()
Y_torch = Y_torch.float()

In [26]:
X_torch.dtype

torch.float32

In [27]:
x_train, x_test, y_train, y_test = train_test_split(X_torch, Y_torch, train_size=0.8)
print(f"Train shape: {x_train.shape}, {y_train.shape}")
print(f"Test shape: {x_test.shape}, {y_test.shape}")


Train shape: torch.Size([960, 5]), torch.Size([960])
Test shape: torch.Size([240, 5]), torch.Size([240])


In [28]:
# Step 4: Create TensorDatasets
train_dataset = TensorDataset(x_train, y_train)
test_dataset = TensorDataset(x_test, y_test)

##
req_batch_size = 32
train_loader = DataLoader(train_dataset, batch_size=req_batch_size, shuffle=True)
test_loader = DataLoader(train_dataset, batch_size=req_batch_size, shuffle=True)

In [29]:
data_iter = iter(train_loader)  # Create an iterator for the DataLoader
x_batch, y_batch = next(data_iter)  # Get the first batch of data
x_batch.dtype

torch.float32

In [31]:
import torch.nn as nn
class myTestLogReg(nn.Module):
    def __init__(self, num_features:int) -> None:
        super().__init__()
        self.fc = nn.Linear(num_features, 1)
        #print(self.fc.parameters.)
        self.sigm = nn.Sigmoid()

    def forward(self, x):
        print(f"Size of the input tensors {x.shape}")
        logreg = self.fc(x)
        probs = self.sigm(logreg)
        return(probs)


In [33]:
modelA = myTestLogReg(num_features = num_feature)
modelA(x_batch)

Size of the input tensors torch.Size([32, 5])


tensor([[0.8334],
        [0.5612],
        [0.4316],
        [0.6768],
        [0.6661],
        [0.4085],
        [0.7618],
        [0.8068],
        [0.5328],
        [0.3271],
        [0.7905],
        [0.5095],
        [0.3816],
        [0.5466],
        [0.6562],
        [0.8030],
        [0.6854],
        [0.5809],
        [0.4463],
        [0.6977],
        [0.5638],
        [0.5280],
        [0.5478],
        [0.5341],
        [0.6829],
        [0.6183],
        [0.4643],
        [0.6976],
        [0.5401],
        [0.3727],
        [0.5937],
        [0.6815]], grad_fn=<SigmoidBackward0>)

In [23]:
test = nn.Linear(3, 1)
test.weight[0].dtype

torch.float32

In [11]:
for batch_x, batch_y in train_loader:
    print(batch_x.shape, batch_y.shape)

torch.Size([32, 5]) torch.Size([32])
torch.Size([32, 5]) torch.Size([32])
torch.Size([32, 5]) torch.Size([32])
torch.Size([32, 5]) torch.Size([32])
torch.Size([32, 5]) torch.Size([32])
torch.Size([32, 5]) torch.Size([32])
torch.Size([32, 5]) torch.Size([32])
torch.Size([32, 5]) torch.Size([32])
torch.Size([32, 5]) torch.Size([32])
torch.Size([32, 5]) torch.Size([32])
torch.Size([32, 5]) torch.Size([32])
torch.Size([32, 5]) torch.Size([32])
torch.Size([32, 5]) torch.Size([32])
torch.Size([32, 5]) torch.Size([32])
torch.Size([32, 5]) torch.Size([32])
torch.Size([32, 5]) torch.Size([32])
torch.Size([32, 5]) torch.Size([32])
torch.Size([32, 5]) torch.Size([32])
torch.Size([32, 5]) torch.Size([32])
torch.Size([32, 5]) torch.Size([32])
torch.Size([32, 5]) torch.Size([32])
torch.Size([32, 5]) torch.Size([32])
torch.Size([32, 5]) torch.Size([32])
torch.Size([32, 5]) torch.Size([32])
torch.Size([32, 5]) torch.Size([32])
torch.Size([32, 5]) torch.Size([32])
torch.Size([32, 5]) torch.Size([32])
t

In [13]:
train_loader.num_workers

0

In [34]:
import pandas as pd
from sklearn.preprocessing import LabelBinarizer

# Sample batch in a pandas DataFrame
data = pd.DataFrame({
    'feature1': [0.2, 0.4, 0.6, 0.8],
    'feature2': [1.0, 1.1, 0.9, 0.7],
    'class': ['cat', 'dog', 'cat', 'rabbit']  # Categorical column
})

# One-hot encoding the target classes
label_binarizer = LabelBinarizer()
target_probs = torch.tensor(label_binarizer.fit_transform(data['class']), dtype=torch.float32)

# Dummy predictions (log probabilities) for KLDivLoss
predicted_log_probs = torch.log(torch.randn(4, 3).softmax(dim=1))  # Log-probabilities


In [35]:
predicted_log_probs

tensor([[-0.3701, -1.4504, -2.5917],
        [-2.4604, -0.1376, -3.1429],
        [-1.1273, -0.7951, -1.4936],
        [-2.2607, -1.3826, -0.4388]])

In [36]:
target_probs

tensor([[1., 0., 0.],
        [0., 1., 0.],
        [1., 0., 0.],
        [0., 0., 1.]])

In [39]:
loss = nn.L1Loss(reduction='none')
input = torch.randn(3, 5, requires_grad=False)
target = torch.randn(3, 5)
loss(input, target).sum(dim=1)

tensor([3.1422, 5.2820, 6.8094])

In [44]:
torch.empty(3, dtype=torch.long).random_(5)

tensor([3, 4, 4])